### 모델 저장과 활용
> 지금까지 만든 모델을 어떻게 남기고、다시 사용하는지 다루어보기 (코드 자체는 2줄)
```python
import joblib

# 저장
joblib.dump(모델, "파일명")

# 불러오기
model = joblib.load("파일명")
```

In [1]:
import os
import time
import joblib
import numpy as pd
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from _style import setup
from _ml import build_dataset, time_split, FEATURES, SEED

setup()
pd.set_option("display.width", 130)

df = build_dataset()
train, test, _ = time_split(df)
tr = train.sample(8000, random_state=SEED)
te = test.sample(2000, random_state=SEED)

X_train, y_train = tr[FEATURES], tr["target_up"]
X_test, y_test = te[FEATURES], te["target_up"]
print(f"학습 {len(tr):,}행／테스트 {len(te):,}행／피처 {len(FEATURES)}개")

학습 8,000행／테스트 2,000행／피처 7개


||학습|예측|
|---|---|---|
|소요 기간|몇 분 ~ 몇 시간|**밀리초(ms)**|
|빈도|주 1회・월 1회|**초당 수백번**|
|필요한 것|전체 학습 데이터|모델 + 입력 하나|

In [ ]:
# 전처리 + 모델 (Pipeline)
pipe = Pipeline([
    ("scaler", StandardScaler()), ("model", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=SEED, n_jobs=-1))
])

# 학습 시간 측정
t0 = time.perf_counter()
pipe.fit(X_train, y_train)
train_s = time.perf_counter() - t0

# 예측 시간 측정 (1건)
single_row = X_test.head(1)
t1 = time.perf_counter()                    # 워밍업: 초기화 비용 측정에서 제외
pipe.predict(single_row)
for _ in range(50):
    pipe.predict(single_row)
predict_s = (time.perf_counter() - t1)/50   # 여러 번(50회) 반복한 후 평균 계산
print(f"학습 1회: {train_s * 1000:.1f}ms")
print(f"예측 1건: {predict_s * 1000:.1f}ms")
print(f"차이: {train_s / predict_s:.0f}배")
print(f"정확도 {pipe.score(X_test, y_test):.4f}")

학습 1회: 268.1ms
예측 1건: 16.3ms
차이: 16배
정확도 0.5005


: 

**요청마다 학습하면 안되는 이유**
> 학습은 예측 1건보다 훨씬 오래 걸리고、데이터가 커질수록 더 길어질 것이다 \
> **학습은 미리 해두고、결과만 꺼내어 사용하는 것**이 저장하는 가장 큰 목적(이유)이다 !